## 4. SVM API 使用 <br>

#### 核心API及参数<br>
在使用支持向量机（SVM）进行分类任务时，常用的API包括`SVC`（支持向量分类器）和`SVR`（支持向量回归器）。<br>
以下是`SVC`的核心API及其最常用参数说明：<br>
- `C`：正则化参数，控制间隔最大化和分类错误最小化的权重。较大的C值会使模型更倾向于最小化分类错误，但可能导致过拟合；较小的C值则更注重间隔最大化。<br>
- `kernel`：核函数类型，常用的有'linear'（线性核）、'poly'（多项式核）、'rbf'（径向基函数核，最常用）和'sigmoid'（Sigmoid核）。选择合适的核函数可以帮助处理非线性分类问题。<br>
- `degree`：多项式核的阶数，仅在`kernel='poly'`时使用。<br>
- `gamma`：核函数的系数，适用于'rbf'、'poly'和'sigmoid'核。较大的gamma值会使模型更复杂，可能导致过拟合；较小的gamma值则使模型更简单。<br>
- `coef0`：核函数中的常数项，适用于'poly'和'sigmoid'核。<br>
- `probability`：是否启用概率估计，默认值为False。启用后，可以使用`predict_proba`方法获取类别概率。<br>
- `max_iter`：最大迭代次数，默认值为-1，表示不限制迭代次数。<br>
- `tol`：停止训练的容忍度，默认值为1e-3。<br>
- `random_state`：随机数生成器的种子，用于保证结果的可重复性。<br>
- `fit_intercept`：是否计算截距，默认值为True。<br>

#### 1. 导入数据集

In [1]:
import pandas as pd
data = pd.read_csv('../../06 Machine Learning/LO2/SVM/Social_Network_Ads.csv')
X = data.iloc[:, :-1].values
y = data.iloc[:, -1].values
X.shape, y.shape

((400, 2), (400,))

#### 2. 数据集划分

In [2]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

#### 3. 参数矩阵<br>
注意：参数的命名规则为：`步骤名称__参数名称`，例如这里的SVM步骤名称为`svm`，所以C参数的名称为`svm__C`。

In [4]:
params = {
    'svm__C': [0.1, 1, 10, 100],
    'svm__kernel': ['linear', 'rbf', 'poly'],
    'svm__gamma': ['scale', 'auto'],
    'svm__degree': [2, 3, 4]  # 仅在kernel为'poly'时有效
}

#### 4. 创建 cv 对象

In [5]:
from sklearn.model_selection import KFold, cross_val_score
cv_strategy = KFold(n_splits=5, shuffle=True, random_state=42)

#### 5. 创建Pipeline, 包含数据预处理和SVM模型

In [3]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC()) # 这里参数留空，因为需要超参数调优，使用GridSearchCV完成
])

#### 6. 使用 GridSearchCV 进行超参数调优

In [6]:
from sklearn.model_selection import GridSearchCV
grid_search = GridSearchCV(
    estimator=pipeline, # 使用上面创建的Pipeline
    param_grid=params, # 使用上面定义的参数矩阵
    scoring='accuracy', # 评估指标为准确率
    cv=cv_strategy, # 使用上面创建的交叉验证策略
    n_jobs=-1, # 使用所有可用的CPU核心进行并行计算
    verbose=2 # 输出详细的日志信息
)

#### 7. 训练模型

In [7]:
grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 72 candidates, totalling 360 fits
[CV] END svm__C=0.1, svm__degree=2, svm__gamma=scale, svm__kernel=linear; total time=   0.0s
[CV] END svm__C=0.1, svm__degree=2, svm__gamma=scale, svm__kernel=linear; total time=   0.0s
[CV] END svm__C=0.1, svm__degree=2, svm__gamma=scale, svm__kernel=poly; total time=   0.0s
[CV] END svm__C=0.1, svm__degree=2, svm__gamma=scale, svm__kernel=rbf; total time=   0.0s
[CV] END svm__C=0.1, svm__degree=2, svm__gamma=scale, svm__kernel=linear; total time=   0.0s
[CV] END svm__C=0.1, svm__degree=2, svm__gamma=scale, svm__kernel=linear; total time=   0.0s
[CV] END svm__C=0.1, svm__degree=2, svm__gamma=scale, svm__kernel=rbf; total time=   0.0s
[CV] END svm__C=0.1, svm__degree=2, svm__gamma=scale, svm__kernel=rbf; total time=   0.0s
[CV] END svm__C=0.1, svm__degree=2, svm__gamma=scale, svm__kernel=rbf; total time=   0.0s
[CV] END svm__C=0.1, svm__degree=2, svm__gamma=scale, svm__kernel=poly; total time=   0.0s
[CV] END svm__C=0.1, svm

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...svm', SVC())])"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'svm__C': [0.1, 1, ...], 'svm__degree': [2, 3, ...], 'svm__gamma': ['scale', 'auto'], 'svm__kernel': ['linear', 'rbf', ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the compu

#### 8. 输出最佳参数和最佳得分和最佳模型

In [8]:
best_params = grid_search.best_params_
best_score = grid_search.best_score_
best_model = grid_search.best_estimator_
print("Best Parameters:", best_params)
print("Best Cross-Validation Score:", best_score)

Best Parameters: {'svm__C': 1, 'svm__degree': 2, 'svm__gamma': 'scale', 'svm__kernel': 'rbf'}
Best Cross-Validation Score: 0.9


#### 9. 在测试集上评估最佳模型

In [9]:
y_pred = best_model.predict(X_test)

#### 10. 评估模型性能

In [10]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
print("Test Set Accuracy:", accuracy)
print("Classification Report:\n", report)
print("Confusion Matrix:\n", conf_matrix)

Test Set Accuracy: 0.93
Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.90      0.94        63
           1       0.86      0.97      0.91        37

    accuracy                           0.93       100
   macro avg       0.92      0.94      0.93       100
weighted avg       0.94      0.93      0.93       100

Confusion Matrix:
 [[57  6]
 [ 1 36]]
